In [7]:
import pandas as pd
import plotly.graph_objects as go
import os
from IPython.display import display

# ==========================================
# 1. HÀM ĐẾM SỐ TỪ TRÙNG NHAU (Word Overlap)
# ==========================================
def count_overlap(words_a, words_b):
    if pd.isna(words_a) or pd.isna(words_b):
        return 0
    set_a = set([w.strip().lower() for w in str(words_a).split(',')])
    set_b = set([w.strip().lower() for w in str(words_b).split(',')])
    return len(set_a.intersection(set_b))

# ==========================================
# 2. LOGIC TÍNH TOÁN VÀ VẼ HEATMAP TRỰC TIẾP
# ==========================================
def compare_all_topics_inline(truth_path, communities_dir):
    print("⏳ Đang tải dữ liệu Topic gốc (Subreddit Thực tế)...")
    if not os.path.exists(truth_path):
        print(f"❌ LỖI: Không tìm thấy file gốc tại: {truth_path}")
        return

    df_truth = pd.read_csv(truth_path)
    subreddits = df_truth['Subreddit'].tolist()
    truth_keywords = df_truth['Top 20 Keywords (LDA)'].tolist()

    algorithms = ['leiden', 'louvain', 'spectral']
    all_best_matches = []

    for algo in algorithms:
        comm_path = os.path.join(communities_dir, f"community_topics_{algo}.csv")

        if not os.path.exists(comm_path):
            print(f"⚠️ Bỏ qua thuật toán {algo.upper()} do không tìm thấy file: {comm_path}")
            continue

        print(f"\n{'='*60}\n🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: {algo.upper()}\n{'='*60}")
        df_algo = pd.read_csv(comm_path)

        communities = df_algo['Community_ID'].tolist()
        algo_keywords = df_algo['Top 20 Keywords (LDA)'].tolist()

        # 2.1 TẠO MA TRẬN ĐIỂM SỐ (Rows: Subreddit, Cols: Community)
        matrix = []
        for sub_kw in truth_keywords:
            row = []
            for comm_kw in algo_keywords:
                overlap_score = count_overlap(sub_kw, comm_kw)
                row.append(overlap_score)
            matrix.append(row)

        # 2.2 TÌM "BEST MATCH"
        for col_idx, comm_id in enumerate(communities):
            col_scores = [matrix[row_idx][col_idx] for row_idx in range(len(subreddits))]
            max_score = max(col_scores)

            if max_score > 0:
                best_sub_idx = col_scores.index(max_score)
                all_best_matches.append({
                    'Algorithm': algo.capitalize(),
                    'Community (ID)': comm_id,
                    'Subreddit (Best Match)': subreddits[best_sub_idx],
                    'Score (Matching words)': f"{max_score}/20",
                    'Keywords Community (Algorithm)': algo_keywords[col_idx],
                    'Keywords Subreddit (Reality)': truth_keywords[best_sub_idx]
                })

        # 2.3 VẼ BIỂU ĐỒ HEATMAP
        # --- Sắp xếp cột X: theo thứ tự số, đổi tên "Cluster" -> "Community" ---
        sorted_comm_indices = sorted(range(len(communities)), key=lambda i: int(communities[i]))
        sorted_communities = [communities[i] for i in sorted_comm_indices]
        # Cập nhật matrix theo thứ tự cột mới
        sorted_matrix = [[row[i] for i in sorted_comm_indices] for row in matrix]
        comm_labels = [f"Community {c}" for c in sorted_communities]

        # --- Sắp xếp hàng Y: tạo đường chéo ---
        # Với mỗi community (cột) theo thứ tự, chọn subreddit chưa được chọn có điểm cao nhất
        assigned = set()
        diagonal_order = []

        for col_idx in range(len(sorted_communities)):
            best_row = None
            best_score = -1
            for row_idx in range(len(subreddits)):
                if row_idx in assigned:
                    continue
                score = sorted_matrix[row_idx][col_idx]
                if score > best_score:
                    best_score = score
                    best_row = row_idx
            if best_row is not None:
                diagonal_order.append(best_row)
                assigned.add(best_row)

        # Thêm các subreddit còn lại chưa được assign vào cuối
        for row_idx in range(len(subreddits)):
            if row_idx not in assigned:
                diagonal_order.append(row_idx)

        sorted_subreddits = [subreddits[i] for i in diagonal_order]
        sorted_matrix = [sorted_matrix[i] for i in diagonal_order]

        fig = go.Figure(data=go.Heatmap(
            z=sorted_matrix,
            x=comm_labels,
            y=sorted_subreddits,
            colorscale='Blues',
            text=sorted_matrix,
            texttemplate="%{text}",
            hoverongaps=False
        ))

        fig.update_layout(
            title=f"Heatmap: Level matching topic - Algorithm {algo.capitalize()}",
            xaxis_title="Algorithm community (Community ID)",
            yaxis_title="Reality community (Subreddit)",
            width=900,
            height=max(400, len(subreddits) * 30)
        )

        fig.show()

    # ==========================================
    # 3. IN BẢNG BÁO CÁO TỔNG HỢP
    # ==========================================
    if all_best_matches:
        print(f"\n{'='*60}\n📋 BẢNG TỔNG HỢP: CÁC CỤM KHỚP NHẤT VỚI THỰC TẾ\n{'='*60}")
        df_report = pd.DataFrame(all_best_matches)
        df_report = df_report.sort_values(by=['Algorithm', 'Score (Matching words)'], ascending=[True, False])
        display(df_report)

        os.makedirs("outputs", exist_ok=True)
        report_path = "outputs/topic_matching_report.csv"
        df_report.to_csv(report_path, index=False)
        print(f"\n💾 (Đã tự động lưu bảng báo cáo này tại: {report_path})")

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN VÀ CHẠY
# ==========================================
TRUTH_PATH = "../../outputs/LDA/subreddit_topics_ground_truth.csv"
COMMUNITIES_DIR = "../../outputs/LDA/"

compare_all_topics_inline(TRUTH_PATH, COMMUNITIES_DIR)

⏳ Đang tải dữ liệu Topic gốc (Subreddit Thực tế)...

🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: LEIDEN



🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: LOUVAIN



🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: SPECTRAL



📋 BẢNG TỔNG HỢP: CÁC CỤM KHỚP NHẤT VỚI THỰC TẾ


,Algorithm,Community (ID),Subreddit (Best Match),Score (Matching words),Keywords Community (Algorithm),Keywords Subreddit (Reality)
2,Leiden,1,Brawlstars,9/20,"new, deck, brawl, brawler, reddit, rules, low,...","brawlstars, questions, removed, rules, brawler..."
19,Leiden,19,NoMansSkyTheGame,4/20,"ship, planet, freighter, save, base, space, bu...","flair, bug, question, bot, ship, answer, answe..."
21,Leiden,21,skyrim,4/20,"skyrim, bugs, said, mods, mod, thats, dont, fp...","skyrim, armor, mod, mods, level, quest, ll, di..."
3,Leiden,6,Destiny,20/20,"trump, right, shit, going, bad, did, actually,...","trump, right, shit, going, did, bad, country, ..."
4,Leiden,7,deadbydaylight,20/20,"killer, survivor, survivors, seconds, killers,...","killer, survivor, survivors, seconds, killers,..."
6,Leiden,4,Battlefield,20/20,"maps, battlefield, bf6, map, better, lol, bad,...","maps, battlefield, bf6, map, better, lol, bad,..."
7,Leiden,5,2007scape,20/20,"lol, content, account, did, doing, new, going,...","content, lol, account, did, doing, new, going,..."
8,Leiden,9,Minecraft,20/20,"comment, minecraft, quality, purpose, downvote...","comment, minecraft, quality, purpose, downvote..."
16,Leiden,14,DotA2,20/20,"dota, hero, team, score, heroes, lane, damage,...","dota, team, hero, score, lane, heroes, damage,..."
10,Leiden,12,ffxiv,19/20,"going, content, ll, new, ddos, level, lot, doe...","content, going, ll, ddos, new, level, lot, act..."



💾 (Đã tự động lưu bảng báo cáo này tại: outputs/topic_matching_report.csv)
